# EduHarness — Colab API Setup（混合路线）

1. 本地：`git archive --format=zip -o eduharness-colab.zip HEAD` → 上传 `MyDrive/eduharness/eduharness-colab.zip`
2. Colab 🔑 **Secrets** 添加名为 **`openai`** 的 key，并开启 **Notebook access**
3. 运行本 notebook；runs 在 `MyDrive/eduharness/runs/`

> 每个 notebook 独立内核；`02`/`03` 会通过 `ensure_api_key()` 自动读取 Secret `openai`。


In [ ]:
from __future__ import annotations
import json, os, sys, subprocess
from getpass import getpass
from pathlib import Path

from google.colab import drive  # type: ignore
drive.mount("/content/drive", force_remount=False)

DRIVE_ZIP = Path("/content/drive/MyDrive/eduharness/eduharness-colab.zip")
REPO = Path("/content")
RUN_DIR = Path("/content/drive/MyDrive/eduharness/runs")
REQUEST_REL = "examples/sample_request.json"

if not DRIVE_ZIP.is_file():
    raise FileNotFoundError(f"请先把 zip 上传到 Drive: {DRIVE_ZIP}")

get_ipython().system(f'unzip -o "{DRIVE_ZIP}" -d /content')

if not (REPO / "educast" / "pipeline.py").is_file():
    raise FileNotFoundError(f"解压后未找到 {REPO}/educast/pipeline.py")

RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.environ["EDUHARNESS_TEXT_MODEL"] = "gpt-5"
os.environ["EDUHARNESS_VISION_MODEL"] = "gpt-5"
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")

# 推荐：Colab 左侧 🔑 Secrets 添加 OPENAI_API_KEY，后续 notebook 会自动读取
from educast.colab_session import ensure_api_key
ensure_api_key()

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements" / "base.txt")])

cfg = {
    "REPO": str(REPO),
    "RUN_DIR": str(RUN_DIR),
    "REQUEST": str(REPO / REQUEST_REL),
}
Path("/content/eduharness_colab_env.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
print(json.dumps(cfg, indent=2, ensure_ascii=False))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


In [ ]:
# 2) 本地衔接（仅提示）
print("""
推荐一键 API 流程：上传并打开 notebooks/06_colab_api_oneshot.ipynb
  PHASE='bootstrap' → 本地 2-render → PHASE='review' → 本地 stage 3

分步（旧）：02_stage1_plan → 03_stage2_render (MODE='prepare')

同步 run：MyDrive/eduharness/runs/<request_id> ↔ 本地 runs/<request_id>
""")
